# Phase 3 — Continuous Decoding · DANDI_000688 (S3)

Predicts hand velocity (vx, vy) from binned spike counts using Ridge, Wiener filter,
and Kalman filter on DANDI_000688 (~54 electrodes, vx ≠ vy confirmed).

| Cell | Content |
|------|---------|
| 1 | Imports + AWS credentials |
| 2 | S3 connection + session load (DANDI_000688) |
| 3 | Feature extraction: binned spikes (X), mean-pooled velocity (y), inter-trial filter |
| 4 | Temporal 80/20 train/test split (`shuffle=False`) |
| 5 | Ridge decoder |
| 6 | Wiener filter decoder |
| 7 | Kalman filter decoder |
| 8 | Summary table |
| 9 | Predicted vs actual velocity plots |

**Prerequisites:** AWS credentials for the `cv-pc` profile must be present in
`~/.aws/credentials`. If the S3 connection fails, set up credentials with
`aws configure --profile cv-pc` or copy the key/secret from the lab's shared
credentials file before running Cell 2.

In [ ]:
from pathlib import Path
import sys
_repo_root = Path.cwd() if (Path.cwd() / "decoding").is_dir() else Path.cwd().parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import os
import configparser
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bci_decoding_dataset import DatasetLoader

from decoding import (
    compute_bin_phases,
    compute_binned_counts,
    DimReducer,
    KalmanFilterDecoder,
    RidgeDecoder,
    WienerFilterDecoder,
)

sns.set_theme(style="whitegrid", font_scale=1.1)
warnings.filterwarnings("ignore", category=PendingDeprecationWarning)

# ── Pipeline parameters (only values that may need tuning) ──────────────────
BIN_SIZE_MS = 50    # spike binning window (ms)
N_LAGS      = 5     # Wiener filter: past bins appended as features
TRAIN_FRAC  = 0.80  # temporal train/test split
PLOT_BINS   = 200   # bins shown in trace plots

# ── AWS credentials from ~/.aws/credentials ─────────────────────────────────
credentials_path = os.path.expanduser("~/.aws/credentials")
config = configparser.ConfigParser()
config.read(credentials_path)
profile = os.environ.get("AWS_PROFILE", "cv-pc")

print("\u2713 All imports successful")
print(f"AWS profile  : {profile}")
print(f"bin_size_ms  : {BIN_SIZE_MS}  |  n_lags: {N_LAGS}  |  train: {int(TRAIN_FRAC*100)}%")

## S3 Connection + Session Load

DANDI_000688 has ~54 electrodes, ~111 sessions, ~180 trials/session.
Velocity has been confirmed correct for this dataset (vx ≠ vy).

In [ ]:
loader = DatasetLoader(
    aws_store=True,
    s3_bucket="solzbacher-lab-motor-decoding-ds",
    s3_key="datasets/Combined_Motor_Datasets",
    aws_access_key_id=config[profile]["aws_access_key_id"],
    aws_secret_access_key=config[profile]["aws_secret_access_key"],
)

sessions  = loader.filter_sessions("dataset_id", "DANDI_000688")
session_id = sessions[0]
ds = loader.get_processed_data_from_session(session_id)

print(f"Session:        {session_id}")
print(f"Subject:        {ds.attrs.get('subject_id', 'unknown')}")
print(f"Task:           {ds.attrs.get('task_type', 'unknown')}")
print(f"Sampling rate:  {ds.attrs.get('sampling_rate', 'unknown')} Hz")
print(f"Spikes shape:   {ds['spikes'].shape}  (n_electrodes \u00d7 n_time)")
print(f"Velocity shape: {ds['velocity'].shape}")
print("\u2713 Connected to S3 and session loaded")

## Feature Extraction

`compute_binned_counts` sums spikes in non-overlapping 50 ms windows → `(n_bins, n_electrodes)`.
Velocity is mean-pooled over the same windows to produce one target per bin.

Inter-trial bins (`trial_phase == 0`) are removed so the decoder trains and evaluates
only on neural states that accompany active task performance.

In [ ]:
X_all      = compute_binned_counts(ds, bin_size_ms=BIN_SIZE_MS)   # (n_bins, n_el)
n_bins     = X_all.shape[0]
bin_phases = compute_bin_phases(ds, bin_size_ms=BIN_SIZE_MS)       # (n_bins,)

velocity  = ds["velocity"].values          # (n_time, 2), float32
n_trimmed = n_bins * BIN_SIZE_MS
y_all = velocity[:n_trimmed].reshape(n_bins, BIN_SIZE_MS, 2).mean(axis=1)  # (n_bins, 2)

active_mask = bin_phases > 0
X = X_all[active_mask]
y = y_all[active_mask]

print(f"All bins:    X={X_all.shape}   y={y_all.shape}")
print(f"Active bins: X={X.shape}   y={y.shape}")
print(f"Dropped {(~active_mask).sum()} inter-trial bins ({(~active_mask).mean()*100:.1f}%)")
print(f"vx range: [{y[:, 0].min():.3f}, {y[:, 0].max():.3f}]")
print(f"vy range: [{y[:, 1].min():.3f}, {y[:, 1].max():.3f}]")
assert not np.allclose(y[:, 0], y[:, 1]), "vx == vy bug detected \u2014 wrong dataset"
print("\u2713 vx \u2260 vy confirmed")

## Train/Test Split

Single temporal cutpoint at 80%: first 80% of active bins → train, last 20% → test.
`shuffle=False` is mandatory — shuffling leaks future neural state into training,
producing optimistically biased R² on time-series data.

In [ ]:
split = int(len(X) * TRAIN_FRAC)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train: {X_train.shape[0]} bins  ({X_train.shape[0] * BIN_SIZE_MS / 1000:.1f} s)")
print(f"Test:  {X_test.shape[0]} bins   ({X_test.shape[0] * BIN_SIZE_MS / 1000:.1f} s)")

## Ridge Decoder

Ridge minimises RSS + L2 penalty on coefficient magnitudes.
The penalty (alpha) prevents overfitting when n_features is comparable to n_bins.
This is the simplest sensible benchmark — every more complex method should beat it.

In [ ]:
ridge = RidgeDecoder(alpha=1.0)
ridge.fit(X_train, y_train)

r2_ridge      = ridge.score(X_test, y_test)   # shape (2,)
y_pred_ridge  = ridge.predict(X_test)

print(f"Ridge  R\u00b2 vx = {r2_ridge[0]:.4f}")
print(f"Ridge  R\u00b2 vy = {r2_ridge[1]:.4f}")

## Wiener Filter Decoder

Extends Ridge by appending the previous `n_lags` bins as extra features,
giving the model `n_lags × 50 ms` of neural history.
Motor cortex activity leads hand velocity by ~100–200 ms, so temporal lags
are the key advantage over plain Ridge.

In [ ]:
wiener = WienerFilterDecoder(n_lags=N_LAGS)
wiener.fit(X_train, y_train)

r2_wiener     = wiener.score(X_test, y_test)   # shape (2,); trims first N_LAGS bins internally
y_pred_wiener = wiener.predict(X_test)          # shape (n_test - N_LAGS, 2)
y_test_wiener = y_test[N_LAGS:]                 # aligned ground truth

print(f"Wiener R\u00b2 vx = {r2_wiener[0]:.4f}  ({N_LAGS} lags = {N_LAGS * BIN_SIZE_MS} ms history)")
print(f"Wiener R\u00b2 vy = {r2_wiener[1]:.4f}")

## Kalman Filter Decoder

Probabilistic state-space decoder that propagates an uncertainty estimate (covariance)
forward in time and uses a Kalman gain to balance the transition prediction against
the current neural observation.
Classical BCI gold standard — should outperform both linear baselines on clean velocity data.

`predict(X_test, y_test)` requires the ground-truth test targets: `y_test[0]` seeds
the initial state estimate only; subsequent steps use only neural observations.

In [ ]:
kalman = KalmanFilterDecoder(C=1, lag=0)
kalman.fit(X_train, y_train)

r2_kalman     = kalman.score(X_test, y_test)          # shape (2,)
y_pred_kalman = kalman.predict(X_test, y_test)

print(f"Kalman R\u00b2 vx = {r2_kalman[0]:.4f}")
print(f"Kalman R\u00b2 vy = {r2_kalman[1]:.4f}")

## Summary

In [ ]:
summary = pd.DataFrame({
    "Decoder": ["Ridge", f"Wiener (n_lags={N_LAGS})", "Kalman"],
    "R\u00b2 vx": [f"{r2_ridge[0]:.4f}", f"{r2_wiener[0]:.4f}", f"{r2_kalman[0]:.4f}"],
    "R\u00b2 vy": [f"{r2_ridge[1]:.4f}", f"{r2_wiener[1]:.4f}", f"{r2_kalman[1]:.4f}"],
    "Features": [
        f"{X.shape[1]} electrodes",
        f"{X.shape[1] * (N_LAGS + 1)} lagged ({N_LAGS + 1} bins \u00d7 {X.shape[1]} el)",
        f"{X.shape[1]} electrodes",
    ],
})
display(summary)

## Visualization — Predicted vs Actual Velocity

First `PLOT_BINS` test bins for each decoder. A good decoder tracks the shape and
sign of the true velocity signal even if it under-estimates the amplitude.

In [ ]:
pb_r = min(PLOT_BINS, len(y_test))
pb_w = min(PLOT_BINS, len(y_test_wiener))
pb_k = min(PLOT_BINS, len(y_test))
t_r  = np.arange(pb_r) * BIN_SIZE_MS / 1000
t_w  = np.arange(pb_w) * BIN_SIZE_MS / 1000
t_k  = np.arange(pb_k) * BIN_SIZE_MS / 1000

def _plot(ax, t, y_true, y_pred, r2_val, title):
    ax.plot(t, y_true, color="#2d6a9f", lw=1.2, label="Actual",    alpha=0.85)
    ax.plot(t, y_pred, color="#e05c2a", lw=1.2, label="Predicted", ls="--", alpha=0.85)
    ax.set_title(f"{title}  (R\u00b2 = {r2_val:.3f})", fontsize=10)
    ax.set_xlabel("Time (s)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_xlim(t[0], t[-1])

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle(
    f"Predicted vs Actual Velocity \u2014 {session_id} (DANDI_000688)\n"
    f"Test set, first {PLOT_BINS} active bins shown",
    fontsize=12,
)

_plot(axes[0, 0], t_r, y_test[:pb_r, 0], y_pred_ridge[:pb_r, 0],   r2_ridge[0],  "Ridge \u2014 vx")
_plot(axes[0, 1], t_r, y_test[:pb_r, 1], y_pred_ridge[:pb_r, 1],   r2_ridge[1],  "Ridge \u2014 vy")
_plot(axes[1, 0], t_w, y_test_wiener[:pb_w, 0], y_pred_wiener[:pb_w, 0], r2_wiener[0], f"Wiener (n_lags={N_LAGS}) \u2014 vx")
_plot(axes[1, 1], t_w, y_test_wiener[:pb_w, 1], y_pred_wiener[:pb_w, 1], r2_wiener[1], f"Wiener (n_lags={N_LAGS}) \u2014 vy")
_plot(axes[2, 0], t_k, y_test[:pb_k, 0], y_pred_kalman[:pb_k, 0], r2_kalman[0], "Kalman \u2014 vx")
_plot(axes[2, 1], t_k, y_test[:pb_k, 1], y_pred_kalman[:pb_k, 1], r2_kalman[1], "Kalman \u2014 vy")

plt.tight_layout()
plt.savefig("01_ContinuousDecoding_DANDI688_velocity.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: 01_ContinuousDecoding_DANDI688_velocity.png")